# 09b – Encode notes with fine-tuned ClinicalBERT

Encodes each note-hour into a 768-d embedding using the fine-tuned ClinicalBERT, producing
the hourly text representation consumed by the fusion models.

**Run after notebooks 07 and 09a** — uses hourly_notes_24h.csv and mbert_best.pt.

**Produces:** text_hourly_cls_mbert.npz (used by notebooks 10, 11).

MIMIC-III data not included (PhysioNet DUA); see README.

## Settings

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import os, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Load the fine-tuned MBERT (BERT weights only)
09a saved a dict of the whole model (bert.* + fc.*). We load a fresh ClinicalBERT
architecture and copy in only the `bert.` weights from the fine-tuned checkpoint.

In [ ]:
CKPT = data_path("mbert_ckpt/mbert_best.pt")
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(device)

# load fine-tuned weights: keep only keys starting with "bert." and strip the prefix
state = torch.load(CKPT, map_location=device, weights_only=True)
bert_state = { k[len("bert."):]: v for k,v in state.items() if k.startswith("bert.") }
missing, unexpected = bert.load_state_dict(bert_state, strict=False)
print("loaded fine-tuned MBERT weights")
print("missing keys:", len(missing), "| unexpected:", len(unexpected))
bert.eval()
for p in bert.parameters(): p.requires_grad = False
HID = bert.config.hidden_size
MAX_LEN = 256

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded fine-tuned MBERT weights
missing keys: 0 | unexpected: 0


## 2. Load notes + cohort

In [ ]:
notes = pd.read_csv(data_path("hourly_notes_24h.csv"))
cohort = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"))
notes["TEXT_CLEAN"] = notes["TEXT_CLEAN"].fillna("")

key = cohort[["HADM_ID","ICUSTAY_ID","mortality_after_24h"]].drop_duplicates()
stay_df = key.drop_duplicates("ICUSTAY_ID").reset_index(drop=True)
notes = notes[notes["HADM_ID"].isin(set(stay_df["HADM_ID"]))].copy()
print("stays:", len(stay_df))

stays: 10068


## 3. Encode each stay-hour's text with frozen MBERT -> [CLS], L2-normalise

In [ ]:
@torch.no_grad()
def encode_texts(texts):
    out=[]; B=16
    for i in range(0,len(texts),B):
        enc = tokenizer(texts[i:i+B], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt").to(device)
        cls = bert(**enc).last_hidden_state[:,0]
        cls = F.normalize(cls, p=2, dim=1)
        out.append(cls.cpu())
    return torch.cat(out,0).numpy() if out else np.zeros((0,HID),dtype='float32')

stay_ids = stay_df["ICUSTAY_ID"].values
hadm_of = stay_df.set_index("ICUSTAY_ID")["HADM_ID"].to_dict()

nz = notes[notes["TEXT_CLEAN"].str.len()>0][["HADM_ID","hour_bin","TEXT_CLEAN"]].reset_index(drop=True)
print("non-empty stay-hours to encode:", len(nz))
emb = encode_texts(nz["TEXT_CLEAN"].tolist())
nz["idx"] = np.arange(len(nz))

X_text = np.zeros((len(stay_ids),24,HID), dtype="float32")
has_note = np.zeros((len(stay_ids),24), dtype="float32")
pos_of_hadm = {hadm_of[s]: i for i,s in enumerate(stay_ids)}
for r in nz.itertuples():
    if r.HADM_ID in pos_of_hadm:
        p = pos_of_hadm[r.HADM_ID]
        X_text[p, int(r.hour_bin)] = emb[r.idx]
        has_note[p, int(r.hour_bin)] = 1.0
print("X_text:", X_text.shape, "| mean note-hours/stay:", round(has_note.sum(1).mean(),2))

non-empty stay-hours to encode: 41238
X_text: (10068, 24, 768) | mean note-hours/stay: 4.1


## 4. Save fine-tuned embeddings

In [ ]:
np.savez_compressed(data_path("text_hourly_cls_mbert.npz"), X_text=X_text, has_note=has_note, stay_ids=stay_ids)
print("saved:", data_path("text_hourly_cls_mbert.npz"))